In [1]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage,AIMessage,SystemMessage
from langchain_ollama import ChatOllama
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver,MemorySaver  # both are same


In [2]:
llm = ChatOllama(model='qwen2.5:0.5b-instruct',temperature=0.2,max_token=100)

In [3]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]
    


In [4]:
def chat_node(state:ChatState):
    messages = state['messages']
    res = llm.invoke(messages)
    return {'messages':[res.content]}


In [5]:
checkpointer = InMemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)


In [6]:
thread_id = '1'
while True:
    user_message = input("Ask anything: ")
    print("Human : ",user_message)

    if user_message.strip().lower() in ['exit','bye','quit']:
        break

    config = {'configurable':{'thread_id':thread_id}}
    res = chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)

    print('AI : ',res['messages'][-1].content)

Human :  exit


In [9]:
config = {'configurable':{'thread_id':thread_id}}
chatbot.get_state(config=config)

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [10]:
for message_chunk,metadata in chatbot.stream({'messages':[HumanMessage(content="hi generate a essay on topic my friend")]},config=config,stream_mode='messages'):
    if message_chunk.content:
        print(message_chunk.content,end=' ',flush=True)

Certainly !  Here 's  an  essay  on  the  topic  " My  Friend ":

 ---

 ** My  Friend :  A  Tale  of  Friendship **

 As  I  sit  here  today ,  surrounded  by  the  familiar  faces  and  memories  that  have  shaped  me  into  who  I  am  today ,  one  person  stands  out  as  a  constant  presence  in  my  life .  This  individual  is  not  just  anyone ;  they  are  my  friend .

 Friend ship  is  a  deep ,  un spoken  bond  where  two  souls  come  together  to  share  their  joys ,  sor rows ,  and  experiences .  It 's  the  kind  of  connection  that  transc ends  time  and  space ,  allowing  us  to  be  present  for  each  other  even  when  we 're  miles  apart .  My  friend ,  let 's  call  them  " Tom ,"  has  been  an  integral  part  of  my  life  since  childhood .

 From  our  earliest  days  together ,  Tom  has  been  a  constant  source  of  laughter  and  support .  We 've  shared  countless  adventures  that  have  made  us  stronger  as  individuals .  Whether  i